In [3]:
import numpy as np
filename = "/home/taeseung/test/PreferenceTransformer/subgoal_vae_hopper-medium-expert-v2.npz"  # 실제 파일
data = np.load(filename, allow_pickle=True)
print(data["params"])
print("Keys in npz:", data.files)  # 예: ['params', 'encoder_Dense_0_bias', ...]

for k in data.files:
    print(k, data[k].shape, data[k].dtype)

['decoder' 'encoder' 'prior']
Keys in npz: ['params']
params (3,) <U7


In [ ]:
import d4rl
import gym
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

class RewardComparison:
    def __init__(self, env_name, reward_model, vae_model):
        """
        Args:
            env_name (str): D4RL 환경 이름 (예: 'hopper-medium-v2')
            reward_model: 학습된 reward model
            vae_model: 학습된 VAE model
        """
        self.env = gym.make(env_name)
        self.dataset = d4rl.qlearning_dataset(self.env)
        self.reward_model = reward_model
        self.vae_model = vae_model
        
        # True rewards from D4RL dataset
        self.true_rewards = self.dataset['rewards']
        
    def generate_predicted_rewards(self, batch_size=1024):
        """
        Reward model과 VAE를 사용하여 예측된 reward 생성
        """
        states = self.dataset['observations']
        actions = self.dataset['actions']
        next_states = self.dataset['next_observations']
        
        reward_model_predictions = []
        vae_predictions = []
        
        # Batch processing
        for i in range(0, len(states), batch_size):
            batch_states = torch.FloatTensor(states[i:i+batch_size])
            batch_actions = torch.FloatTensor(actions[i:i+batch_size])
            batch_next_states = torch.FloatTensor(next_states[i:i+batch_size])
            
            # Reward model predictions
            with torch.no_grad():
                reward_pred = self.reward_model(batch_states, batch_actions)
                reward_model_predictions.append(reward_pred.cpu().numpy())
                
                # VAE predictions
                vae_pred = self.vae_model.predict_reward(batch_states, batch_actions, batch_next_states)
                vae_predictions.append(vae_pred.cpu().numpy())
        
        self.reward_model_predictions = np.concatenate(reward_model_predictions)
        self.vae_predictions = np.concatenate(vae_predictions)
        
    def calculate_metrics(self):
        """
        다양한 메트릭을 계산하여 reward 예측의 정확도 평가
        """
        metrics = {}
        
        # MSE
        metrics['rm_mse'] = mean_squared_error(self.true_rewards, self.reward_model_predictions)
        metrics['vae_mse'] = mean_squared_error(self.true_rewards, self.vae_predictions)
        
        # MAE
        metrics['rm_mae'] = mean_absolute_error(self.true_rewards, self.reward_model_predictions)
        metrics['vae_mae'] = mean_absolute_error(self.true_rewards, self.vae_predictions)
        
        # Correlation coefficients
        metrics['rm_pearson'], _ = pearsonr(self.true_rewards, self.reward_model_predictions)
        metrics['vae_pearson'], _ = pearsonr(self.true_rewards, self.vae_predictions)
        metrics['rm_spearman'], _ = spearmanr(self.true_rewards, self.reward_model_predictions)
        metrics['vae_spearman'], _ = spearmanr(self.true_rewards, self.vae_predictions)
        
        return metrics
    
    def plot_comparisons(self):
        """
        True reward와 예측된 reward들을 시각화
        """
        plt.figure(figsize=(15, 5))
        
        # Scatter plots
        plt.subplot(131)
        plt.scatter(self.true_rewards, self.reward_model_predictions, alpha=0.5, label='Reward Model')
        plt.scatter(self.true_rewards, self.vae_predictions, alpha=0.5, label='VAE')
        plt.plot([min(self.true_rewards), max(self.true_rewards)], 
                [min(self.true_rewards), max(self.true_rewards)], 
                'k--', label='Perfect Prediction')
        plt.xlabel('True Rewards')
        plt.ylabel('Predicted Rewards')
        plt.legend()
        plt.title('Reward Predictions vs True Rewards')
        
        # Distribution plot
        plt.subplot(132)
        plt.hist(self.true_rewards, bins=50, alpha=0.5, label='True', density=True)
        plt.hist(self.reward_model_predictions, bins=50, alpha=0.5, label='Reward Model', density=True)
        plt.hist(self.vae_predictions, bins=50, alpha=0.5, label='VAE', density=True)
        plt.xlabel('Reward Values')
        plt.ylabel('Density')
        plt.legend()
        plt.title('Reward Distributions')
        
        # Time series plot
        plt.subplot(133)
        sample_idx = np.arange(100)  # Plot first 100 samples
        plt.plot(sample_idx, self.true_rewards[sample_idx], label='True', alpha=0.7)
        plt.plot(sample_idx, self.reward_model_predictions[sample_idx], label='Reward Model', alpha=0.7)
        plt.plot(sample_idx, self.vae_predictions[sample_idx], label='VAE', alpha=0.7)
        plt.xlabel('Time Steps')
        plt.ylabel('Reward Values')
        plt.legend()
        plt.title('Reward Time Series')
        
        plt.tight_layout()
        plt.show()

# 사용 예시
def main():
    # 환경과 모델 설정
    env_name = "hopper-medium-expert-v2"
    reward_model = YourRewardModel()  # 사용자의 reward model
    vae_model = YourVAEModel()        # 사용자의 VAE model
    
    # 비교 실험 실행
    comparison = RewardComparison(env_name, reward_model, vae_model)
    comparison.generate_predicted_rewards()
    
    # 메트릭 계산
    metrics = comparison.calculate_metrics()
    print("Performance Metrics:")
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")
    
    # 시각화
    comparison.plot_comparisons()

if __name__ == "__main__":
    main()